# Chroma CRUD Operations

This notebook walks through create, read, update, and delete operations with a local Chroma vector store.

In [7]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings

## 1. Set Up Paths and the Vector Store

In [4]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('g:/sushant/Advanced_RAG/4.Vector_Store')

In [ ]:
# Load environment variables from the local .env file.
dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("Please add your GROQ_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: g:\sushant\Advanced_RAG\4.Vector_Store\.env


In [9]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo"
persist_directory = project_root / "db" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo
Persist directory: g:\sushant\Advanced_RAG\4.Vector_Store\db\chroma_langchain_db


In [10]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

No previous Chroma directory was found.


In [ ]:
# Create the embedding model and connect it to a persistent Chroma store.
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = OllamaEmbeddings(model="qwen3-embedding:4b")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings, ## embedding model
    persist_directory=str(persist_directory), ### vector store persistence path in disk
)

print("Vector store is ready.")

Vector store is ready.


## 2. Add Small Helper Functions

In [12]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Create and Insert Example Documents

In [13]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [14]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [16]:
print(uuid4())  ### generates a random 128 bits UUID for each document to be used as the document ID in the vector store

9c630c8b-8ef8-474a-991e-aa078fa74abf


In [17]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1. id=b05a476e-a9fc-49ee-b2a6-4c1c20d2ef4a
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=f9837252-4efb-4b37-a6a2-e6c83434af36
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=2f19f4d9-f5cd-46b1-b077-f4c7f274b85c
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=72c925ad-460b-410c-ae87-0fd04f76bdfc
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=b9b5c0d7-906c-4b52-9883-ef8e599958a8
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=9fc35e5e-5707-4637-a495-c23af7ff169d
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [18]:
documents[0].id

'b05a476e-a9fc-49ee-b2a6-4c1c20d2ef4a'

In [19]:
### Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
b05a476e-a9fc-49ee-b2a6-4c1c20d2ef4a
f9837252-4efb-4b37-a6a2-e6c83434af36
2f19f4d9-f5cd-46b1-b077-f4c7f274b85c
72c925ad-460b-410c-ae87-0fd04f76bdfc
b9b5c0d7-906c-4b52-9883-ef8e599958a8
9fc35e5e-5707-4637-a495-c23af7ff169d
920bb721-aee7-441c-81aa-cef99e9fad7e
75db02db-33cc-4580-ac3d-6040b00a27e8
1a0828ae-1cc5-4cf4-8858-8a9740c09fe3
81ecd6c9-2f97-4538-8379-9994c2e5302b

Total inserted documents: 10


## 4. Read the Stored Data Back

In [20]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [21]:
raw_records

{'ids': ['b05a476e-a9fc-49ee-b2a6-4c1c20d2ef4a',
  'f9837252-4efb-4b37-a6a2-e6c83434af36',
  '2f19f4d9-f5cd-46b1-b077-f4c7f274b85c',
  '72c925ad-460b-410c-ae87-0fd04f76bdfc',
  'b9b5c0d7-906c-4b52-9883-ef8e599958a8',
  '9fc35e5e-5707-4637-a495-c23af7ff169d',
  '920bb721-aee7-441c-81aa-cef99e9fad7e',
  '75db02db-33cc-4580-ac3d-6040b00a27e8',
  '1a0828ae-1cc5-4cf4-8858-8a9740c09fe3',
  '81ecd6c9-2f97-4538-8379-9994c2e5302b'],
 'embeddings': array([[-0.00020475,  0.00099121, -0.00875847, ..., -0.01054727,
          0.0252891 , -0.00269049],
        [-0.00033769, -0.00243409, -0.0093718 , ..., -0.00822426,
          0.00144852, -0.00164744],
        [-0.00030063,  0.03907269, -0.04193546, ..., -0.04337408,
         -0.00316057, -0.00019648],
        ...,
        [-0.00015885,  0.03112788, -0.01342704, ..., -0.01205513,
          0.00302185, -0.00119325],
        [-0.00042902, -0.04212976, -0.01056797, ...,  0.0035374 ,
          0.02336858, -0.01676279],
        [-0.00048786, -0.01906164, 

In [22]:
print(raw_records["embeddings"].shape)

(10, 2560)


In [ ]:
print(raw_records["embeddings"][0:2, 0:20]) ### for first two documents, show the first 20 dimensions of the embedding vector. 

[[-0.00020475  0.00099121 -0.00875847 -0.0265531  -0.00128079  0.07258503
   0.03958801 -0.04281564  0.01243988 -0.02937148  0.03956015  0.0122359
   0.0024844  -0.09400713  0.01823334 -0.01073462  0.0271143  -0.01259523
  -0.00401501 -0.00251197]
 [-0.00033769 -0.00243409 -0.0093718  -0.00175008 -0.00170499  0.05264548
   0.04600117 -0.03044066  0.01427812 -0.05233341  0.06465575  0.02592979
   0.00030403 -0.09766168  0.02112567 -0.00530067  0.02309022  0.00187896
  -0.00508644 -0.00426913]]


In [40]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
73a1c0e2-e6d2-40c2-a128-6d244087f245
f8efa688-bc5a-4a32-a2e6-c7d1806b0a28
ff63b73b-3685-4373-840b-ce0ed60b448c


In [27]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[:3]
selected_ids

['b05a476e-a9fc-49ee-b2a6-4c1c20d2ef4a',
 'f9837252-4efb-4b37-a6a2-e6c83434af36',
 '2f19f4d9-f5cd-46b1-b077-f4c7f274b85c']

In [28]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=b05a476e-a9fc-49ee-b2a6-4c1c20d2ef4a
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=f9837252-4efb-4b37-a6a2-e6c83434af36
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=2f19f4d9-f5cd-46b1-b077-f4c7f274b85c
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.



In [29]:
print(selected_documents)

[Document(id='b05a476e-a9fc-49ee-b2a6-4c1c20d2ef4a', metadata={'topic': 'AI', 'doc_number': 1}, page_content='Artificial intelligence helps machines perform tasks that usually need human reasoning.'), Document(id='f9837252-4efb-4b37-a6a2-e6c83434af36', metadata={'doc_number': 2, 'topic': 'AI'}, page_content='AI systems can analyze patterns in data to support predictions and automation.'), Document(id='2f19f4d9-f5cd-46b1-b077-f4c7f274b85c', metadata={'topic': 'AI', 'doc_number': 3}, page_content='Responsible AI development includes fairness, transparency, and safety checks.')]


## 5. Run a Similarity Search

In [30]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [31]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=72c925ad-460b-410c-ae87-0fd04f76bdfc
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=b9b5c0d7-906c-4b52-9883-ef8e599958a8
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
3. id=9fc35e5e-5707-4637-a495-c23af7ff169d
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they make semantic search over embedded documents possible.



In [32]:
search_results

[Document(id='72c925ad-460b-410c-ae87-0fd04f76bdfc', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='b9b5c0d7-906c-4b52-9883-ef8e599958a8', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
 Document(id='9fc35e5e-5707-4637-a495-c23af7ff169d', metadata={'topic': 'RAG', 'doc_number': 6}, page_content='Vector stores are important in RAG because they make semantic search over embedded documents possible.')]

In [ ]:
vector_store.similarity_search_with_score(query=query, k=4)  ## returns tuple with Document object and euclidian distance score between query vector and document embedding vector for each of the top k results in float.

[(Document(id='72c925ad-460b-410c-ae87-0fd04f76bdfc', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.24619661271572113),
 (Document(id='b9b5c0d7-906c-4b52-9883-ef8e599958a8', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  0.30940619111061096),
 (Document(id='9fc35e5e-5707-4637-a495-c23af7ff169d', metadata={'topic': 'RAG', 'doc_number': 6}, page_content='Vector stores are important in RAG because they make semantic search over embedded documents possible.'),
  0.5563647150993347),
 (Document(id='75db02db-33cc-4580-ac3d-6040b00a27e8', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.6956198811531067)]

## 6. Update Existing Documents

In [34]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['72c925ad-460b-410c-ae87-0fd04f76bdfc',
 '75db02db-33cc-4580-ac3d-6040b00a27e8']

In [35]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=72c925ad-460b-410c-ae87-0fd04f76bdfc
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=75db02db-33cc-4580-ac3d-6040b00a27e8
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [36]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [37]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
72c925ad-460b-410c-ae87-0fd04f76bdfc
75db02db-33cc-4580-ac3d-6040b00a27e8


In [38]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=72c925ad-460b-410c-ae87-0fd04f76bdfc
metadata={'doc_number': 4, 'topic': 'RAG'}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=75db02db-33cc-4580-ac3d-6040b00a27e8
metadata={'doc_number': 8, 'topic': 'LLM'}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [39]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [41]:
updated_search_results = vector_store.similarity_search(updated_query, k=3)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=72c925ad-460b-410c-ae87-0fd04f76bdfc
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=b9b5c0d7-906c-4b52-9883-ef8e599958a8
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
3. id=9fc35e5e-5707-4637-a495-c23af7ff169d
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they make semantic search over embedded documents possible.



## 7. Delete Documents

In [42]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['1a0828ae-1cc5-4cf4-8858-8a9740c09fe3',
 '81ecd6c9-2f97-4538-8379-9994c2e5302b']

In [43]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
1a0828ae-1cc5-4cf4-8858-8a9740c09fe3
81ecd6c9-2f97-4538-8379-9994c2e5302b


In [44]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
b05a476e-a9fc-49ee-b2a6-4c1c20d2ef4a
f9837252-4efb-4b37-a6a2-e6c83434af36
2f19f4d9-f5cd-46b1-b077-f4c7f274b85c
72c925ad-460b-410c-ae87-0fd04f76bdfc
b9b5c0d7-906c-4b52-9883-ef8e599958a8
9fc35e5e-5707-4637-a495-c23af7ff169d
920bb721-aee7-441c-81aa-cef99e9fad7e
75db02db-33cc-4580-ac3d-6040b00a27e8

Deleted ids still present?
1a0828ae-1cc5-4cf4-8858-8a9740c09fe3: False
81ecd6c9-2f97-4538-8379-9994c2e5302b: False


In [45]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
